# MuJoCo WASM — separate-assets prototype

Run the smoke test first. MuJoCo physics runs in the browser; Python provides scene XML and low-frequency commands.

First, install dependencies:

In [1]:
# Install Python dependencies and download the viewer files.
import os
import sys
import subprocess

REPO_URL = "https://github.com/szafird/mujoco-example.git"
REPO_DIR = "/content/mujoco_course"

if "google.colab" in sys.modules:
    # Clone the repository containing the widget and assets.
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             REPO_URL, REPO_DIR],
            check=True
        )

    # Install dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "-r", f"{REPO_DIR}/requirements.txt"],
        check=True
    )

    # Make the widget module importable.
    sys.path.insert(0, REPO_DIR)

    # Enable interactive Jupyter widgets in Colab.
    from google.colab import output
    output.enable_custom_widget_manager()

    print("MuJoCo viewer setup complete!")

else:
    print("Running outside Colab; using local project files.")

Running outside Colab; using local project files.


## 1. Configure GitHub Pages asset URL

Replace `YOUR_USER` and `YOUR_REPO` after enabling Pages from the `docs/` directory. Use the trailing slash.


In [2]:
ASSET_BASE = "https://szafird.github.io/mujoco-example/mujoco/"

## 2. MuJoCo-only initialization test (no 3D scene)

If this reports SUCCESS, the unmodified JS loader and WASM binary are working together.


In [3]:
from IPython.display import display
from mujoco_widget import MuJoCoSmokeTest, MuJoCoViewer
smoke = MuJoCoSmokeTest(asset_base=ASSET_BASE)
display(smoke)


In [4]:
print(smoke.status)


SUCCESS: MuJoCo WASM initialized (3.17.0)


## 3. Live scene (after smoke test passes)


In [ ]:
scene_xml = '<mujoco model="Browser demo"><option timestep="0.005"/><worldbody><light pos="0 0 4"/><geom type="plane" size="3 3 0.1" rgba="0.8 0.85 0.9 1"/><body pos="0 0 1.6"><freejoint/><geom type="sphere" size="0.18" mass="1" rgba="0.95 0.3 0.2 1"/></body><body pos="0.6 0 1.2"><freejoint/><geom type="box" size="0.2 0.2 0.2" mass="1" rgba="0.2 0.55 0.85 1"/></body></worldbody></mujoco>'
viewer = MuJoCoViewer(scene_xml, asset_base=ASSET_BASE)
display(viewer)
print("test")


## 4. Python commands


In [6]:
viewer.pause()
viewer.reset()
viewer.set_gravity(-3.0)
viewer.play()


In [7]:
print(viewer.status, viewer.sim_time)


Ready (drag to orbit, scroll to zoom) 2.549999999999968


## Python feedback-control API (V3)
Run the cells below after the viewer displays **Ready**. Requests use `await` because the physics engine runs in your browser.


In [8]:
viewer.pause()
state = await viewer.get_sim_state()
print("time:", state["time"])
print("qpos:", state["qpos"])
print("qvel:", state["qvel"])


TimeoutError: 

In [ ]:
viewer.reset()
viewer.pause()
state = await viewer.step(100)
print("Time after 100 steps:", state["time"])
print("Positions:", state["qpos"])


### Example: Python-driven simulation loop
The falling objects have no actuators, so this demonstrates stepping and state inspection, not closed-loop control. For a controller exercise we will add a motorized pendulum or cart-pole.


In [ ]:
viewer.reset()
viewer.pause()
samples = []
for _ in range(20):
    state = await viewer.step(10)
    samples.append((state["time"], state["qpos"][2]))
print("First five (time, ball z) samples:", samples[:5])
